# Shot Caption Pipeline — FPT Cloud VLM

Notebook tạo caption tiếng Việt cho từng shot và xuất **một file SQL duy nhất**.

## Input

- `videos.txt`: mỗi dòng `video_id<TAB>video_url` (cũng chấp nhận dấu phẩy hoặc khoảng trắng đầu tiên).
- `shots.csv`: các cột `shot_id,video_id,shot_index,start_ms,end_ms,start_frame_idx,end_frame_idx`.
- FPT API key: lưu trong **Kaggle Secrets** với nhãn `FPT_API_KEY`; không ghi key trực tiếp vào notebook.

Lát cắt tuân theo quy ước Python `[SLICE_START:SLICE_END)`. Ví dụ `0:8000` xử lý đúng 8.000 shot đầu tiên. Hai input nên được sort sẵn; notebook vẫn kiểm tra tính hợp lệ và giữ nguyên thứ tự của `shots.csv`.

## Thiết kế song song và resume

1. Tải các video cần cho lát cắt bằng `DOWNLOAD_WORKERS` worker.
2. `SHOT_WORKERS` worker trích nhiều frame ứng viên, lọc ảnh mờ/tối/trùng và chọn keyframe đa dạng. Shot dài được ghép thành storyboard; số request đồng thời được giới hạn riêng bởi `API_WORKERS`.
3. Chỉ luồng chính ghi checkpoint và SQL, vì nhiều worker append chung một file có thể làm hỏng hoặc đảo thứ tự output.
4. Chạy lại cùng lát cắt sẽ bỏ qua các shot đã có trong checkpoint. SQL cuối luôn được dựng lại theo thứ tự input.

FPT Dedicated Inference dùng API tương thích Chat Completions. Với endpoint Marketplace hoặc endpoint riêng, hãy điền URL đầy đủ kết thúc bằng `/v1/chat/completions` và đúng model name được cấp.

In [ ]:
from __future__ import annotations

import base64
import csv
import json
import os
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any
from urllib.parse import unquote, urlparse

import cv2
import requests

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **_: Any):
        return iterable if iterable is not None else None

print("OpenCV:", cv2.__version__)
print("requests:", requests.__version__)

## 1. Cấu hình

Điền đường dẫn dataset Kaggle và model/endpoint FPT. `CAPTION_ID_OFFSET=1` khiến shot ở dòng toàn cục thứ 0 có `caption_id=1`; do đó các notebook chạy các lát cắt khác nhau không trùng ID. Nếu database thực tế dùng identity và không cho insert ID thủ công, đổi `SQL_INSERT_TEMPLATE` cho phù hợp schema khi import.

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    videos_file: Path = Path("/kaggle/input/aic-caption-input/videos.txt")
    shots_file: Path = Path("/kaggle/input/aic-caption-input/shots.csv")
    work_dir: Path = Path("/kaggle/working/caption_pipeline")
    output_sql: Path = Path("/kaggle/working/captions.sql")
    slice_start: int = 0
    slice_end: int | None = 8000
    caption_id_offset: int = 1
    pipeline_version: str = "adaptive_storyboard_v1"

    fpt_endpoint: str = "https://mkp-api.fptcloud.com/v1/chat/completions"
    fpt_model: str = "Qwen2.5-VL-7B-Instruct"
    api_key_secret_name: str = "FPT_API_KEY"

    download_workers: int = 4
    shot_workers: int = 8
    api_workers: int = 4
    request_timeout_s: int = 120
    max_retries: int = 5
    max_candidate_frames: int = 16
    max_keyframes: int = 8
    short_shot_ms: int = 2000
    medium_shot_ms: int = 6000
    duplicate_distance: float = 0.10
    min_blur_score: float = 25.0
    storyboard_columns: int = 3
    storyboard_tile_width: int = 448
    storyboard_tile_height: int = 252
    jpeg_quality: int = 88
    max_tokens: int = 350
    temperature: float = 0.1

    overwrite_videos: bool = False
    resume: bool = True

CONFIG = PipelineConfig()
RUN_PRODUCTION = False  # Đổi thành True sau khi đã kiểm tra cấu hình.

CAPTION_PROMPT = """Bạn là hệ thống mô tả shot video phục vụ truy vấn video tiếng Việt.
Bạn nhận một hoặc hai ảnh từ cùng shot. Mỗi ảnh có thể là frame đơn hoặc storyboard.
Trong storyboard, đọc các ô theo thời gian từ trái sang phải, từ trên xuống dưới; bỏ qua ô đen đệm cuối lưới.
Nếu có ảnh frame đơn đi kèm storyboard, dùng nó để quan sát chi tiết và không xem là thời điểm mới.
Hãy viết DUY NHẤT một đoạn tiếng Việt dài 2-4 câu, mô tả chi tiết những gì quan sát được:
- bối cảnh, không gian, thời điểm/ánh sáng nếu thấy rõ;
- người, động vật, đồ vật nổi bật, đặc điểm và vị trí tương đối;
- hành động, chuyển động, tương tác và thay đổi qua các ảnh;
- góc nhìn hoặc chuyển động máy quay nếu nhận biết chắc chắn.
Không OCR, không chép hoặc đoán chữ xuất hiện trong ảnh. Không suy đoán danh tính, địa điểm,
nguyên nhân, cảm xúc hay sự kiện ngoài bằng chứng thị giác. Không dùng bullet, JSON, tiêu đề hay lời dẫn."""

SQL_INSERT_TEMPLATE = (
    "INSERT INTO Caption (caption_id, shot_id, caption_text) "
    "VALUES ({caption_id}, {shot_id}, {caption_text});"
)

print(asdict(CONFIG))

In [ ]:
REQUIRED_SHOT_COLUMNS = {
    "shot_id", "video_id", "shot_index", "start_ms", "end_ms",
    "start_frame_idx", "end_frame_idx",
}

@dataclass(frozen=True)
class Shot:
    global_index: int
    shot_id: str
    video_id: str
    shot_index: int
    start_ms: int
    end_ms: int
    start_frame_idx: int | None
    end_frame_idx: int | None


def _optional_int(value: Any) -> int | None:
    text = "" if value is None else str(value).strip()
    return None if not text else int(text)


def load_video_urls(path: Path) -> dict[str, str]:
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy videos file: {path}")
    result: dict[str, str] = {}
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t", 1)
        if len(parts) == 1:
            parts = line.split(",", 1)
        if len(parts) == 1:
            parts = line.split(maxsplit=1)
        if len(parts) != 2 or not all(part.strip() for part in parts):
            raise ValueError(f"{path}:{line_number}: cần '<video_id><TAB><URL>'")
        video_id, url = (part.strip() for part in parts)
        if video_id in result:
            raise ValueError(f"video_id bị lặp trong {path}: {video_id}")
        parsed = urlparse(url)
        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            raise ValueError(f"URL không hợp lệ tại {path}:{line_number}: {url}")
        result[video_id] = url
    if not result:
        raise ValueError(f"{path} không có video hợp lệ")
    return result


def load_shot_slice(path: Path, start: int, end: int | None) -> list[Shot]:
    if start < 0 or (end is not None and end < start):
        raise ValueError(f"Khoảng không hợp lệ: [{start}:{end})")
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy shots file: {path}")
    shots: list[Shot] = []
    seen_ids: set[str] = set()
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = REQUIRED_SHOT_COLUMNS - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"{path} thiếu cột: {sorted(missing)}")
        for global_index, row in enumerate(reader):
            if global_index < start:
                continue
            if end is not None and global_index >= end:
                break
            shot = Shot(
                global_index=global_index,
                shot_id=row["shot_id"].strip(),
                video_id=row["video_id"].strip(),
                shot_index=int(row["shot_index"]),
                start_ms=int(row["start_ms"]),
                end_ms=int(row["end_ms"]),
                start_frame_idx=_optional_int(row["start_frame_idx"]),
                end_frame_idx=_optional_int(row["end_frame_idx"]),
            )
            if not shot.shot_id or not shot.video_id:
                raise ValueError(f"Dòng shot {global_index + 2}: ID rỗng")
            if shot.shot_id in seen_ids:
                raise ValueError(f"shot_id bị lặp trong lát cắt: {shot.shot_id}")
            if shot.shot_index < 0 or shot.start_ms < 0 or shot.end_ms <= shot.start_ms:
                raise ValueError(f"Shot không hợp lệ: {shot}")
            if (shot.start_frame_idx is None) != (shot.end_frame_idx is None):
                raise ValueError(f"Shot phải có cả hai frame index hoặc cùng để trống: {shot.shot_id}")
            if shot.start_frame_idx is not None and shot.end_frame_idx < shot.start_frame_idx:
                raise ValueError(f"Frame range không hợp lệ: {shot.shot_id}")
            seen_ids.add(shot.shot_id)
            shots.append(shot)
    if not shots:
        raise ValueError(f"Không có shot trong lát cắt [{start}:{end})")
    return shots


def safe_video_path(video_dir: Path, video_id: str, url: str) -> Path:
    safe_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", video_id).strip("._")
    if not safe_id:
        raise ValueError(f"video_id không thể tạo filename an toàn: {video_id!r}")
    suffix = Path(unquote(urlparse(url).path)).suffix.lower()
    if not re.fullmatch(r"\.[a-z0-9]{1,5}", suffix):
        suffix = ".mp4"
    return video_dir / f"{safe_id}{suffix}"


def download_video(video_id: str, url: str, target: Path, overwrite: bool = False) -> Path:
    if target.is_file() and target.stat().st_size > 0 and not overwrite:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".part")
    try:
        with requests.get(url, stream=True, timeout=(20, 300)) as response:
            response.raise_for_status()
            with partial.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        if partial.stat().st_size == 0:
            raise RuntimeError(f"Video tải về rỗng: {video_id}")
        partial.replace(target)
        return target
    finally:
        if partial.exists():
            partial.unlink()


def download_required_videos(
    shots: list[Shot], video_urls: dict[str, str], config: PipelineConfig
) -> dict[str, Path]:
    video_ids = list(dict.fromkeys(shot.video_id for shot in shots))
    missing = [video_id for video_id in video_ids if video_id not in video_urls]
    if missing:
        raise KeyError(f"Thiếu URL cho video_id: {missing[:10]}")
    video_dir = config.work_dir / "videos"
    paths = {
        video_id: safe_video_path(video_dir, video_id, video_urls[video_id])
        for video_id in video_ids
    }
    with ThreadPoolExecutor(max_workers=config.download_workers) as executor:
        futures = {
            executor.submit(
                download_video, video_id, video_urls[video_id], paths[video_id],
                config.overwrite_videos,
            ): video_id
            for video_id in video_ids
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Download videos"):
            future.result()
    return paths

In [ ]:
@dataclass(frozen=True)
class FrameCandidate:
    timestamp_ms: int
    frame: Any
    sharpness: float
    brightness: float


def adaptive_candidate_count(shot: Shot, config: PipelineConfig) -> int:
    duration_ms = shot.end_ms - shot.start_ms
    if duration_ms <= config.short_shot_ms:
        return min(3, config.max_candidate_frames)
    # Xấp xỉ một ứng viên mỗi 1,5 giây, có chặn để thời gian decode ổn định.
    return min(config.max_candidate_frames, max(6, (duration_ms + 1499) // 1500))


def sample_frame_times_ms(shot: Shot, count: int) -> list[int]:
    if count < 1:
        raise ValueError("candidate frame count phải >= 1")
    duration = shot.end_ms - shot.start_ms
    # Tâm của các đoạn đều nhau, không chạm end_ms (thường là biên exclusive).
    return [shot.start_ms + int(duration * (index + 0.5) / count) for index in range(count)]


def frame_quality(frame: Any) -> tuple[float, float]:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return sharpness, float(gray.mean())


def extract_frame_candidates(
    video_path: Path, shot: Shot, config: PipelineConfig
) -> list[FrameCandidate]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Không mở được video {video_path} cho shot {shot.shot_id}")
    candidates: list[FrameCandidate] = []
    try:
        count = adaptive_candidate_count(shot, config)
        for timestamp_ms in sample_frame_times_ms(shot, count):
            capture.set(cv2.CAP_PROP_POS_MSEC, float(timestamp_ms))
            ok, frame = capture.read()
            if not ok or frame is None:
                continue
            sharpness, brightness = frame_quality(frame)
            candidates.append(FrameCandidate(timestamp_ms, frame, sharpness, brightness))
    finally:
        capture.release()
    if not candidates:
        raise RuntimeError(f"Không trích được frame nào cho shot {shot.shot_id}")
    return candidates


def visual_distance(first: Any, second: Any) -> float:
    size = (160, 90)
    first_small = cv2.resize(first, size, interpolation=cv2.INTER_AREA)
    second_small = cv2.resize(second, size, interpolation=cv2.INTER_AREA)
    first_hsv = cv2.cvtColor(first_small, cv2.COLOR_BGR2HSV)
    second_hsv = cv2.cvtColor(second_small, cv2.COLOR_BGR2HSV)
    hist_first = cv2.calcHist([first_hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    hist_second = cv2.calcHist([second_hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    cv2.normalize(hist_first, hist_first)
    cv2.normalize(hist_second, hist_second)
    histogram_distance = float(cv2.compareHist(hist_first, hist_second, cv2.HISTCMP_BHATTACHARYYA))
    pixel_distance = min(1.0, float(cv2.absdiff(first_small, second_small).mean()) / 32.0)
    return 0.55 * histogram_distance + 0.45 * pixel_distance


def filter_low_quality_and_duplicates(
    candidates: list[FrameCandidate], config: PipelineConfig
) -> list[FrameCandidate]:
    usable = [
        candidate for candidate in candidates
        if config.min_blur_score <= candidate.sharpness and 8.0 <= candidate.brightness <= 247.0
    ]
    if not usable:
        usable = candidates
    deduplicated: list[FrameCandidate] = []
    for candidate in usable:
        if not deduplicated:
            deduplicated.append(candidate)
            continue
        if visual_distance(deduplicated[-1].frame, candidate.frame) < config.duplicate_distance:
            if candidate.sharpness > deduplicated[-1].sharpness:
                deduplicated[-1] = candidate
        else:
            deduplicated.append(candidate)
    return deduplicated


def select_diverse_keyframes(
    candidates: list[FrameCandidate], shot: Shot, config: PipelineConfig
) -> list[FrameCandidate]:
    candidates = filter_low_quality_and_duplicates(candidates, config)
    duration_ms = shot.end_ms - shot.start_ms
    if len(candidates) == 1:
        return candidates
    if duration_ms <= config.short_shot_ms:
        midpoint = (shot.start_ms + shot.end_ms) / 2
        return [min(candidates, key=lambda item: abs(item.timestamp_ms - midpoint))]
    if duration_ms <= config.medium_shot_ms:
        best_pair = max(
            ((first, second) for index, first in enumerate(candidates) for second in candidates[index + 1:]),
            key=lambda pair: visual_distance(pair[0].frame, pair[1].frame)
            + 0.2 * abs(pair[1].timestamp_ms - pair[0].timestamp_ms) / duration_ms,
        )
        return list(best_pair)

    target = min(config.max_keyframes, len(candidates))
    selected = [candidates[0], candidates[-1]]
    while len(selected) < target:
        selected_timestamps = {item.timestamp_ms for item in selected}
        remaining = [item for item in candidates if item.timestamp_ms not in selected_timestamps]
        if not remaining:
            break
        next_item = max(
            remaining,
            key=lambda item: min(visual_distance(item.frame, chosen.frame) for chosen in selected)
            + 0.2 * min(abs(item.timestamp_ms - chosen.timestamp_ms) for chosen in selected) / duration_ms,
        )
        selected.append(next_item)
    return sorted(selected, key=lambda item: item.timestamp_ms)


def letterbox_frame(frame: Any, width: int, height: int) -> Any:
    source_height, source_width = frame.shape[:2]
    scale = min(width / source_width, height / source_height)
    resized_width = max(1, int(source_width * scale))
    resized_height = max(1, int(source_height * scale))
    resized = cv2.resize(frame, (resized_width, resized_height), interpolation=cv2.INTER_AREA)
    left = (width - resized_width) // 2
    right = width - resized_width - left
    top = (height - resized_height) // 2
    bottom = height - resized_height - top
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(0, 0, 0))


def build_storyboard(keyframes: list[FrameCandidate], config: PipelineConfig) -> Any:
    columns = min(config.storyboard_columns, len(keyframes))
    tiles = [
        letterbox_frame(item.frame, config.storyboard_tile_width, config.storyboard_tile_height)
        for item in keyframes
    ]
    blank = tiles[0].copy()
    blank[:] = 0
    while len(tiles) % columns:
        tiles.append(blank.copy())
    rows = [cv2.hconcat(tiles[index:index + columns]) for index in range(0, len(tiles), columns)]
    return cv2.vconcat(rows)


def prepare_vlm_images(
    video_path: Path, shot: Shot, config: PipelineConfig
) -> tuple[list[Any], list[FrameCandidate]]:
    keyframes = select_diverse_keyframes(extract_frame_candidates(video_path, shot, config), shot, config)
    if len(keyframes) <= 2:
        return [item.frame for item in keyframes], keyframes
    storyboard = build_storyboard(keyframes, config)
    detail = max(keyframes, key=lambda item: item.sharpness).frame
    return [storyboard, detail], keyframes


def frame_to_data_url(frame: Any, jpeg_quality: int) -> str:
    ok, encoded = cv2.imencode(
        ".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality]
    )
    if not ok:
        raise RuntimeError("Không encode được frame sang JPEG")
    payload = base64.b64encode(encoded.tobytes()).decode("ascii")
    return f"data:image/jpeg;base64,{payload}"


def extract_caption_content(payload: dict[str, Any]) -> str:
    try:
        content = payload["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError(f"Response FPT không đúng Chat Completions: {payload}") from exc
    if isinstance(content, str):
        caption = content.strip()
    elif isinstance(content, list):
        caption = "\n".join(
            str(item.get("text", "")).strip()
            for item in content
            if isinstance(item, dict) and item.get("type") in {"text", "output_text"}
        ).strip()
    else:
        caption = ""
    if not caption:
        raise RuntimeError(f"FPT trả caption rỗng: {payload}")
    return caption


def call_fpt_vlm(
    frames: list[Any], api_key: str, config: PipelineConfig, api_semaphore: threading.Semaphore
) -> str:
    content: list[dict[str, Any]] = [{"type": "text", "text": CAPTION_PROMPT}]
    content.extend(
        {"type": "image_url", "image_url": {"url": frame_to_data_url(frame, config.jpeg_quality)}}
        for frame in frames
    )
    request_payload = {
        "model": config.fpt_model,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": config.max_tokens,
        "temperature": config.temperature,
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    last_error: Exception | None = None
    for attempt in range(config.max_retries):
        try:
            with api_semaphore:
                response = requests.post(
                    config.fpt_endpoint,
                    headers=headers,
                    json=request_payload,
                    timeout=(20, config.request_timeout_s),
                )
            if response.status_code in {408, 429, 500, 502, 503, 504}:
                raise requests.HTTPError(
                    f"FPT HTTP {response.status_code}: {response.text[:500]}", response=response
                )
            response.raise_for_status()
            return extract_caption_content(response.json())
        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_error = exc
            if attempt + 1 == config.max_retries:
                break
            retry_after = None
            if isinstance(exc, requests.HTTPError) and exc.response is not None:
                retry_after = exc.response.headers.get("Retry-After")
            delay = float(retry_after) if retry_after and retry_after.isdigit() else min(30.0, 2**attempt)
            time.sleep(delay + random.uniform(0.0, 0.5))
    raise RuntimeError(f"Gọi FPT thất bại sau {config.max_retries} lần: {last_error}") from last_error


def process_shot(
    shot: Shot,
    video_path: Path,
    api_key: str,
    config: PipelineConfig,
    api_semaphore: threading.Semaphore,
) -> dict[str, Any]:
    api_images, keyframes = prepare_vlm_images(video_path, shot, config)
    caption = call_fpt_vlm(api_images, api_key, config, api_semaphore)
    return {
        "global_index": shot.global_index,
        "caption_id": config.caption_id_offset + shot.global_index,
        "shot_id": shot.shot_id,
        "caption_text": caption,
        "model": config.fpt_model,
        "keyframe_count": len(keyframes),
        "api_image_count": len(api_images),
    }


def load_checkpoint(path: Path) -> dict[str, dict[str, Any]]:
    results: dict[str, dict[str, Any]] = {}
    if not path.is_file():
        return results
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                results[item["shot_id"]] = item
            except (json.JSONDecodeError, KeyError) as exc:
                raise ValueError(f"Checkpoint hỏng tại {path}:{line_number}") from exc
    return results


def append_checkpoint(path: Path, item: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def sql_literal(value: str) -> str:
    # PostgreSQL standard string literal: nháy đơn được nhân đôi; NUL bị cấm.
    if "\x00" in value:
        raise ValueError("SQL text không được chứa NUL")
    return "'" + value.replace("'", "''") + "'"


def write_sql(path: Path, shots: list[Shot], results: dict[str, dict[str, Any]]) -> None:
    missing = [shot.shot_id for shot in shots if shot.shot_id not in results]
    if missing:
        raise RuntimeError(f"Chưa đủ caption nên không xuất SQL; thiếu {len(missing)} shot")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write("BEGIN;\n")
        for shot in shots:
            item = results[shot.shot_id]
            handle.write(SQL_INSERT_TEMPLATE.format(
                caption_id=int(item["caption_id"]),
                shot_id=sql_literal(item["shot_id"]),
                caption_text=sql_literal(item["caption_text"]),
            ) + "\n")
        handle.write("COMMIT;\n")
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def get_fpt_api_key(secret_name: str) -> str:
    key = os.getenv(secret_name, "").strip()
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret(secret_name).strip()
        except (ImportError, Exception) as exc:
            raise RuntimeError(
                f"Không lấy được secret {secret_name!r}. Hãy tạo Kaggle Secret hoặc env var cùng tên."
            ) from exc
    if not key:
        raise RuntimeError(f"Secret {secret_name!r} rỗng")
    return key


def validate_config(config: PipelineConfig) -> None:
    positive = {
        "download_workers": config.download_workers,
        "shot_workers": config.shot_workers,
        "api_workers": config.api_workers,
        "request_timeout_s": config.request_timeout_s,
        "max_retries": config.max_retries,
        "max_candidate_frames": config.max_candidate_frames,
        "max_keyframes": config.max_keyframes,
        "short_shot_ms": config.short_shot_ms,
        "medium_shot_ms": config.medium_shot_ms,
        "storyboard_columns": config.storyboard_columns,
        "storyboard_tile_width": config.storyboard_tile_width,
        "storyboard_tile_height": config.storyboard_tile_height,
    }
    invalid = {name: value for name, value in positive.items() if value < 1}
    if invalid:
        raise ValueError(f"Cấu hình phải dương: {invalid}")
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", config.pipeline_version):
        raise ValueError("pipeline_version chỉ được chứa chữ, số, '.', '_' hoặc '-'")
    if config.max_keyframes < 2:
        raise ValueError("max_keyframes phải >= 2")
    if config.short_shot_ms > config.medium_shot_ms:
        raise ValueError("short_shot_ms không được lớn hơn medium_shot_ms")
    if not 0.0 <= config.duplicate_distance <= 1.0:
        raise ValueError("duplicate_distance phải trong [0, 1]")
    if not 1 <= config.jpeg_quality <= 100:
        raise ValueError("jpeg_quality phải trong [1, 100]")
    if "REPLACE_WITH" in config.fpt_model:
        raise ValueError("Hãy điền đúng fpt_model trước khi chạy")
    parsed = urlparse(config.fpt_endpoint)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError("fpt_endpoint phải là HTTPS URL hợp lệ")


def run_pipeline(config: PipelineConfig, api_key: str | None = None) -> Path:
    validate_config(config)
    config.work_dir.mkdir(parents=True, exist_ok=True)
    shots = load_shot_slice(config.shots_file, config.slice_start, config.slice_end)
    video_urls = load_video_urls(config.videos_file)
    video_paths = download_required_videos(shots, video_urls, config)
    api_key = api_key or get_fpt_api_key(config.api_key_secret_name)

    checkpoint = config.work_dir / (
        f"checkpoint_{config.pipeline_version}_{config.slice_start}_{config.slice_end}.jsonl"
    )
    results = load_checkpoint(checkpoint) if config.resume else {}
    selected_ids = {shot.shot_id for shot in shots}
    results = {shot_id: item for shot_id, item in results.items() if shot_id in selected_ids}
    pending = [shot for shot in shots if shot.shot_id not in results]
    print(f"Selected={len(shots)}, resumed={len(results)}, pending={len(pending)}")

    failures: list[tuple[str, str]] = []
    api_semaphore = threading.Semaphore(config.api_workers)
    with ThreadPoolExecutor(max_workers=config.shot_workers) as executor:
        futures = {
            executor.submit(
                process_shot, shot, video_paths[shot.video_id], api_key, config, api_semaphore
            ): shot
            for shot in pending
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Caption shots"):
            shot = futures[future]
            try:
                item = future.result()
                results[shot.shot_id] = item
                append_checkpoint(checkpoint, item)
            except Exception as exc:
                failures.append((shot.shot_id, repr(exc)))

    if failures:
        failure_path = config.work_dir / (
            f"failures_{config.pipeline_version}_{config.slice_start}_{config.slice_end}.json"
        )
        failure_path.write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
        preview = "\n".join(f"- {shot_id}: {error}" for shot_id, error in failures[:5])
        raise RuntimeError(
            f"{len(failures)} shot thất bại; checkpoint đã giữ kết quả thành công.\n{preview}\n"
            f"Chi tiết: {failure_path}"
        )

    write_sql(config.output_sql, shots, results)
    print(f"Hoàn tất {len(shots)} shot -> {config.output_sql}")
    return config.output_sql

## 2. Chạy production

Cell không chạy mặc định để tránh gọi API nhầm khi cấu hình còn placeholder. Sau khi sửa `CONFIG`, đặt `RUN_PRODUCTION=True`.

In [ ]:
if RUN_PRODUCTION:
    production_sql = run_pipeline(CONFIG)
    print(production_sql.read_text(encoding="utf-8")[:2000])
else:
    print("Production chưa chạy. Sửa CONFIG và đặt RUN_PRODUCTION=True.")

## 3. Mock test thật (cell cuối)

Cell này tải một MP4 công khai, tạo input mock gồm một shot 0–10 giây để kiểm tra cả chọn keyframe/storyboard, gọi **FPT VLM thật**, rồi in SQL. Nó mặc định tắt để không tiêu quota ngoài ý muốn. Trước khi bật:

1. Điền `CONFIG.fpt_model` và nếu cần thì `CONFIG.fpt_endpoint`.
2. Tạo Kaggle Secret `FPT_API_KEY`.
3. Đặt `RUN_MOCK_TEST=True` rồi chạy cell.

Video mock: Google sample video bucket, chỉ dùng để kiểm tra luồng download → lấy frame → API → SQL.

In [1]:
RUN_MOCK_TEST = False

if RUN_MOCK_TEST:
    mock_dir = Path("/kaggle/working/caption_mock")
    mock_dir.mkdir(parents=True, exist_ok=True)
    mock_videos = mock_dir / "videos.txt"
    mock_shots = mock_dir / "shots.csv"
    mock_videos.write_text(
        "MOCK_V001\thttps://storage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4\n",
        encoding="utf-8",
    )
    mock_shots.write_text(
        "shot_id,video_id,shot_index,start_ms,end_ms,start_frame_idx,end_frame_idx\n"
        "MOCK_S000,MOCK_V001,0,0,10000,0,249\n",
        encoding="utf-8",
    )
    mock_config = replace(
        CONFIG,
        videos_file=mock_videos,
        shots_file=mock_shots,
        work_dir=mock_dir / "work",
        output_sql=mock_dir / "mock_captions.sql",
        slice_start=0,
        slice_end=1,
        download_workers=2,
        shot_workers=1,
        api_workers=1,
        resume=False,
    )
    mock_sql = run_pipeline(mock_config)
    print("\n--- MOCK SQL ---")
    print(mock_sql.read_text(encoding="utf-8"))
else:
    print("Mock test chưa chạy. Điền model/secret rồi đặt RUN_MOCK_TEST=True.")

Mock test chưa chạy. Điền model/secret rồi đặt RUN_MOCK_TEST=True.
